In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import json
from pathlib import Path

language_pt = True

# PubMedQA labeled (PQAL) — faça upload no Drive antes de rodar
PQAL_PATH = "/content/drive/MyDrive/FIAP-POS-TECH/ori_pqal.json"

if language_pt:
  OUTPUT_PATH_DATASET = "/content/drive/MyDrive/FIAP-POS-TECH/pqal_train_pt-br.jsonl"
  LORA_OUTPUT_DIR = "/content/drive/MyDrive/FIAP-POS-TECH/ori_pqal-model-pt"
else:
  # Troque para ori_pqal_pt-br.json se preferir português
  OUTPUT_PATH_DATASET = "/content/drive/MyDrive/FIAP-POS-TECH/pqal_train.jsonl"
  LORA_OUTPUT_DIR = "/content/drive/MyDrive/FIAP-POS-TECH/ori_pqal-model"

INSTRUCTION = (
    "Com base no contexto científico fornecido, responda à pergunta "
    "com sim, não ou talvez e justifique brevemente."
)
DECISION_PT = {"yes": "sim", "no": "não", "maybe": "talvez"}

In [8]:

def pqal_to_alpaca(pqal_path: str, out_path: str) -> None:
    with open(pqal_path, encoding="utf-8") as f:
        data = json.load(f)

    records = []
    for entry in data.values():
        context = "\n\n".join(entry["CONTEXTS"])
        decision = DECISION_PT.get(entry["final_decision"].strip().lower(), entry["final_decision"])
        records.append({
            "instruction": INSTRUCTION,
            "input": f"Pergunta: {entry['QUESTION']}\n\nContexto:\n{context}",
            "output": f"Resposta: {decision}\n\n{entry['LONG_ANSWER']}",
        })

    Path(out_path).write_text(
        "\n".join(json.dumps(r, ensure_ascii=False) for r in records),
        encoding="utf-8",
    )
    print(f"Gerados {len(records)} exemplos em {out_path}")

pqal_to_alpaca(PQAL_PATH, OUTPUT_PATH_DATASET)

Gerados 1000 exemplos em /content/drive/MyDrive/FIAP-POS-TECH/pqal_train_pt-br.jsonl


In [9]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install transformers datasets

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-11ohfoxq/unsloth_a2e73be420384a5cb8a288ff3e5991e3
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-11ohfoxq/unsloth_a2e73be420384a5cb8a288ff3e5991e3
  Resolved https://github.com/unslothai/unsloth.git to commit 8760d44374e5f094707d5c0e9e75560200db26b9
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [10]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments, TextStreamer

max_seq_length = 2048
dtype = None
load_in_4bit = True
fourbit_models = [
    "unsloth/mistral-7b-v0.3-bnb-4bit",
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/llama-3-8b-bnb-4bit",
    "unsloth/llama-3-8b-Instruct-bnb-4bit",
    "unsloth/llama-3-70b-bnb-4bit",
    "unsloth/Phi-3-mini-4k-instruct",
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit",
]

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [11]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",

    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.9.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):

        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

In [ ]:
dataset = load_dataset("json", data_files=OUTPUT_PATH_DATASET, split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer, # Add tokenizer back here
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 400,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 4 | Total steps = 400
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.727711
2,1.845244
3,1.682524
4,1.716824
5,1.722757
6,1.597162
7,1.532406
8,1.545274
9,1.477972
10,1.510394


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-400/tokenizer_config.json.


In [ ]:
FastLanguageModel.for_inference(model)

sample_input = (
    "Pergunta: A vacina contra gripe reduz hospitalizações em idosos?\n\n"
    "Contexto: Estudo randomizado com 12.000 idosos mostrou redução de 40% "
    "nas hospitalizações por pneumonia após vacinação anual."
)

inputs = tokenizer(
[
    alpaca_prompt.format(INSTRUCTION, sample_input, ""),
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

Both `max_new_tokens` (=64) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


['<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nCom base no contexto científico fornecido, responda à pergunta com sim, não ou talvez e justifique brevemente.\n\n### Input:\nPergunta: A vacina contra gripe reduz hospitalizações em idosos?\n\nContexto: Estudo randomizado com 12.000 idosos mostrou redução de 40% nas hospitalizações por pneumonia após vacinação anual.\n\n### Response:\nResposta: sim\n\nA vacina contra a gripe reduz a mortalidade e a morbidade em idosos. Estudos observacionais sugeriram que a vacinação reduz as hospitalizações por pneumonia. Um estudo randomizado de 12.000 idosos com idade > ou = ']

In [ ]:
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)


model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Com base no contexto científico fornecido, responda à pergunta com sim, não ou talvez e justifique brevemente.

### Input:
Pergunta: A vacina contra gripe reduz hospitalizações em idosos?

Contexto: Estudo randomizado com 12.000 idosos mostrou redução de 40% nas hospitalizações por pneumonia após vacinação anual.

### Response:


Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Resposta: sim

A vacinação anual contra a gripe é uma estratégia eficaz para reduzir a carga de doença e mortalidade por pneumonias.<|end_of_text|>


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/FIAP-POS-TECH/ori_pqal-model-pt/tokenizer_config.json.


('/content/drive/MyDrive/FIAP-POS-TECH/ori_pqal-model-pt/tokenizer_config.json',
 '/content/drive/MyDrive/FIAP-POS-TECH/ori_pqal-model-pt/tokenizer.json')

In [ ]:
if True:

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = LORA_OUTPUT_DIR,
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load /content/drive/MyDrive/FIAP-POS-TECH/ori_pqal-model-pt as a legacy tokenizer.


In [ ]:
# Carrega um exemplo real do PQAL para teste após o treino
with open(PQAL_PATH, encoding="utf-8") as f:
    pqal_sample = next(iter(json.load(f).values()))

test_input = (
    f"Casso a informação não existe, responder que não possui esse tipo de conhecimento\n\n"
    f"Pergunta: tenho um carro de formula 1?\n\n"
    f""
)

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""


inputs = tokenizer(
[
    alpaca_prompt.format(INSTRUCTION, test_input, ""),
], return_tensors = "pt").to("cuda")

In [ ]:
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Com base no contexto científico fornecido, responda à pergunta com sim, não ou talvez e justifique brevemente.

### Input:
Casso a informação não existe, responder que não possui esse tipo de conhecimento

Pergunta: tenho um carro de formula 1?



### Response:


Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Resposta: não

A maioria dos pacientes não sabe que existe um tipo de conhecimento, a saber, a informação não existente. Acreditam que o conhecimento é uma propriedade objetiva de um conceito, que pode ser adquirido, perdido e adquirido novamente. Essa concepção do conhecimento é refutada pela teoria da memória de autonoma. A informação não existente é um tipo especial de conhecimento que não pode ser adquirido nem perdido. Os pacientes podem ser treinados para reconhecer quando o conhecimento é inform
